# Correlation, the Matched Filter and Energy Detection

This notebook investigates how a signal buried in noise can be found, and how the decision "signal present or not" is made with a controlled false-alarm rate.

## Objective

The objectives of this notebook are to:

- Use cross-correlation to find a known waveform that is invisible in the raw samples.
- Measure the processing gain of the matched filter and compare it with theory.
- See how the waveform shape (rectangular pulse, Barker code, LFM chirp) controls resolution and sidelobes after matched filtering.
- Set a detection threshold from a desired probability of false alarm, and see how sensitive it is to the noise power estimate.
- Compare the matched filter and the energy detector in terms of detection probability and robustness to frequency offset.

## Theory

### Complex Baseband Model

All signals in this notebook are complex baseband (I/Q) samples, as they come out of an SDR. The received signal is

$$
x[n] = s[n - n_0] + w[n]
$$

- $s[n]$ : known waveform of $N$ samples, energy $E = \sum |s[n]|^2$
- $n_0$ : unknown delay
- $w[n]$ : complex white Gaussian noise, $\mathbb{E}\{|w[n]|^2\} = \sigma^2$ (half in I, half in Q)

The per-sample input SNR is $\mathrm{SNR_{in}} = |s[n]|^2 / \sigma^2$ for a constant-envelope waveform.

### Cross-Correlation

Cross-correlation slides the template over the received signal:

$$
r[k] = \sum_{n=0}^{N-1} x[n + k]\, s^*[n]
$$

When $k = n_0$ the template lines up with the signal and all $N$ terms add **in phase**; the noise terms add with random phase.

### Matched Filter

Among all linear filters, the one that maximizes the output SNR at the decision instant in white noise is

$$
h[n] = s^*[N - 1 - n]
$$

i.e. the time-reversed conjugate of the waveform. Filtering with $h$ is the same operation as correlating with $s$. The peak output SNR is

$$
\mathrm{SNR_{out}} = \frac{E}{\sigma^2} = N \cdot \mathrm{SNR_{in}}
\qquad\Rightarrow\qquad
G_p = 10 \log_{10} N \ \text{dB}
$$

Only the **energy** matters for peak SNR, not the shape. The shape decides what the output looks like around the peak (main-lobe width and sidelobes). A long waveform with a wide bandwidth $B$ compresses to a peak of width about $1/B$: this is **pulse compression**.

### Detection as a Hypothesis Test

$$
\mathcal{H}_0 : x[n] = w[n]
\qquad
\mathcal{H}_1 : x[n] = s[n] + w[n]
$$

A test statistic $T$ is compared with a threshold $\gamma$:

$$
P_{FA} = P(T > \gamma \mid \mathcal{H}_0),
\qquad
P_D = P(T > \gamma \mid \mathcal{H}_1)
$$

In the Neyman-Pearson approach, $\gamma$ is fixed from the allowed $P_{FA}$, and $P_D$ is whatever results.

### Matched Filter Detector (unknown phase)

With a unit-energy template $\tilde h = s / \sqrt{E}$, let $y = \sum x[n] \tilde h^*[n]$ and $T_{MF} = |y|^2$.

- Under $\mathcal{H}_0$: $T_{MF}/\sigma^2$ is exponential with mean 1, so $P_{FA} = e^{-\gamma/\sigma^2}$ and $\gamma = -\sigma^2 \ln P_{FA}$.
- Under $\mathcal{H}_1$: $2T_{MF}/\sigma^2$ is non-central chi-square with 2 degrees of freedom and non-centrality $2E/\sigma^2$.

### Energy Detector

When the waveform is unknown, only its energy can be used:

$$
T_{ED} = \sum_{n=0}^{N-1} |x[n]|^2
$$

- Under $\mathcal{H}_0$: $2T_{ED}/\sigma^2$ is chi-square with $2N$ degrees of freedom.
- Under $\mathcal{H}_1$: non-central chi-square with $2N$ degrees of freedom and non-centrality $2E/\sigma^2$.

The energy detector also collects the noise of all $N$ samples, so it needs considerably more SNR than the matched filter. In exchange, it does not care about the waveform, its phase, or its frequency offset.

### Threshold and Noise Power

Both thresholds are proportional to $\sigma^2$. In practice $\sigma^2$ must be **estimated**, and any error in it changes the real $P_{FA}$. This is the motivation for adaptive (CFAR) thresholds.

## Setup

Common imports, sampling parameters, and helper functions for noise generation, waveforms, and Monte Carlo trials.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, stats

# Sampling frequency (complex baseband)
fs = 1e6  # Hz

# Reproducible randomness
rng = np.random.default_rng(0)


def db(x):
    """Power ratio to dB."""
    return 10 * np.log10(np.maximum(x, 1e-30))


def awgn(shape, noise_power=1.0):
    """Complex white Gaussian noise with E{|w|^2} = noise_power."""
    return np.sqrt(noise_power / 2) * (
        rng.standard_normal(shape) + 1j * rng.standard_normal(shape)
    )


def bpsk_code(n_chips, samples_per_chip=1):
    """Random +/-1 code, each chip repeated samples_per_chip times."""
    chips = rng.choice([-1.0, 1.0], size=n_chips)
    return np.repeat(chips, samples_per_chip).astype(complex)


def lfm_chirp(n_samples, bandwidth):
    """Complex LFM chirp sweeping from -B/2 to +B/2 over n_samples."""
    T = n_samples / fs
    t = np.arange(n_samples) / fs - T / 2
    k = bandwidth / T  # sweep rate [Hz/s]
    return np.exp(1j * np.pi * k * t**2)


def run_trials(template, n_trials, rx_signal=None, noise_power=1.0, chunk=5000):
    """
    Monte Carlo trials with the receiver window aligned to the signal.

    Returns the matched filter statistic |y|^2 and the energy detector
    statistic sum |x|^2 for every trial. If rx_signal is None, only noise
    is received (H0).
    """
    h = template / np.linalg.norm(template)  # unit-energy template
    T_mf, T_ed = [], []

    for start in range(0, n_trials, chunk):
        m = min(chunk, n_trials - start)
        x = awgn((m, len(template)), noise_power)
        if rx_signal is not None:
            x = x + rx_signal

        T_mf.append(np.abs(x @ h.conj()) ** 2)
        T_ed.append(np.sum(np.abs(x) ** 2, axis=1))

    return np.concatenate(T_mf), np.concatenate(T_ed)

## Experiment 1: Finding a Signal You Cannot See

Hide a 128-sample random BPSK waveform in a 3000-sample noise record at an input SNR of -6 dB per sample.

Look at the raw samples, then cross-correlate the record with the known waveform and try to read the delay from the correlation output.

In [ ]:
N = 128
snr_in_db = -6
noise_power = 1.0

amp = np.sqrt(noise_power * 10 ** (snr_in_db / 10))
s = amp * bpsk_code(N)

record_len = 3000
true_delay = 1700  # samples

x = awgn(record_len, noise_power)
x[true_delay:true_delay + N] += s

E = np.sum(np.abs(s) ** 2)
print(f"Input SNR per sample:     {snr_in_db} dB")
print(f"Signal energy E/sigma^2:  {E / noise_power:.1f} ({db(E / noise_power):.1f} dB)")
print(f"True delay:               {true_delay} samples ({true_delay / fs * 1e6:.0f} us)")

In [ ]:
# Correlate with the unit-energy template ("valid": template fully inside the record)
h = s / np.linalg.norm(s)
r = signal.correlate(x, h, mode="valid")
lags = np.arange(len(r))

r_power = np.abs(r) ** 2 / noise_power
est_delay = np.argmax(r_power)

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

axes[0].plot(np.arange(record_len), np.abs(x) ** 2, linewidth=0.6)
axes[0].axvspan(true_delay, true_delay + N, color="tab:red", alpha=0.2, label="true signal location")
axes[0].set_xlabel("Sample index")
axes[0].set_ylabel("|x[n]|^2")
axes[0].set_title("Received samples (instantaneous power)")
axes[0].grid()
axes[0].legend()

axes[1].plot(lags, r_power, linewidth=0.8)
axes[1].axvline(true_delay, color="tab:red", linestyle=":", label="true delay")
axes[1].set_xlabel("Lag [samples]")
axes[1].set_ylabel("|r[k]|^2 / sigma^2")
axes[1].set_title("Cross-correlation with the known waveform")
axes[1].grid()
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Estimated delay:          {est_delay} samples")
print(f"Peak / sigma^2:           {db(r_power[est_delay]):.1f} dB")
print(f"Largest noise-only value: {db(np.max(np.delete(r_power, np.arange(est_delay - N, est_delay + N)))):.1f} dB")

### Observation

- 

## Experiment 2: Processing Gain

Measure the output SNR of the matched filter for waveform lengths from 16 to 1024 samples, keeping the input SNR fixed at -10 dB per sample.

The output SNR is measured over many trials as $|\mathbb{E}\{y\}|^2 / \mathrm{Var}\{y\}$ at the correct delay.

As a comparison, apply a **mismatched** filter of the same length: a plain moving average (all-ones template), which does not know the BPSK code.

In [ ]:
lengths = [16, 32, 64, 128, 256, 512, 1024]
snr_in_db = -10
amp = np.sqrt(10 ** (snr_in_db / 10))
n_trials = 4000

gain_mf, gain_avg = [], []

print(f"{'N':>6} {'theory [dB]':>12} {'matched [dB]':>13} {'moving avg [dB]':>16}")
print("-" * 51)

for N in lengths:
    s = amp * bpsk_code(N)
    x = awgn((n_trials, N)) + s

    h_mf = s / np.linalg.norm(s)
    h_avg = np.ones(N) / np.sqrt(N)

    for h, out in [(h_mf, gain_mf), (h_avg, gain_avg)]:
        y = x @ h.conj()
        snr_out = np.abs(np.mean(y)) ** 2 / np.var(y)
        out.append(db(snr_out) - snr_in_db)

    print(f"{N:>6} {db(N):>12.2f} {gain_mf[-1]:>13.2f} {gain_avg[-1]:>16.2f}")

In [ ]:
plt.figure(figsize=(9, 5))
plt.semilogx(lengths, db(np.array(lengths)), "k--", label="theory: 10 log10(N)")
plt.semilogx(lengths, gain_mf, "o-", label="matched filter (measured)")
plt.semilogx(lengths, gain_avg, "s-", label="moving average (measured)")
plt.xlabel("Waveform length N [samples]")
plt.ylabel("SNR_out - SNR_in [dB]")
plt.title("Processing gain")
plt.grid(which="both")
plt.legend()
plt.show()

### Observation

- 

## Experiment 3: Waveform Shape and Pulse Compression

Compare three waveforms with the **same duration (130 µs) and the same energy**:

- **Rectangular pulse:** constant envelope, no modulation.
- **Barker-13:** 13 chips of 10 µs each, phase-coded with the Barker sequence.
- **LFM chirp:** frequency sweep over 100 kHz.

First look at each autocorrelation (the matched filter output for an isolated echo). Then receive two echoes only 20 µs apart and check which waveform can separate them.

In [ ]:
n_samples = 130  # 130 us at 1 MHz

barker13 = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1], dtype=complex)

waveforms = {
    "Rectangular": np.ones(n_samples, dtype=complex),
    "Barker-13": np.repeat(barker13, 10),
    "LFM chirp (B = 100 kHz)": lfm_chirp(n_samples, 100e3),
}

# Normalize all waveforms to unit energy
waveforms = {k: v / np.linalg.norm(v) for k, v in waveforms.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, s in waveforms.items():
    acf = signal.correlate(s, s, mode="full")
    lags = signal.correlation_lags(len(s), len(s), mode="full")
    acf_db = db(np.abs(acf) ** 2)

    axes[0].plot(lags / fs * 1e6, acf_db, label=name)
    axes[1].plot(lags / fs * 1e6, np.abs(acf), label=name)

    # -3 dB main-lobe width and peak sidelobe level
    main = acf_db >= -3
    width_us = np.sum(main) / fs * 1e6
    peak = np.argmax(acf_db)
    # walk out of the main lobe to the first null/minimum on the right side
    i = peak
    while i + 1 < len(acf_db) and acf_db[i + 1] < acf_db[i]:
        i += 1
    psl = f"{np.max(acf_db[i:]):6.1f} dB" if i < len(acf_db) - 1 else "  none"
    print(f"{name:<25} -3 dB width = {width_us:6.1f} us | peak sidelobe = {psl}")

axes[0].set_xlabel("Lag [us]")
axes[0].set_ylabel("Normalized |R|^2 [dB]")
axes[0].set_title("Autocorrelation (dB)")
axes[0].set_ylim(-50, 3)
axes[0].grid()
axes[0].legend()

axes[1].set_xlabel("Lag [us]")
axes[1].set_ylabel("Normalized |R|")
axes[1].set_title("Autocorrelation (linear)")
axes[1].grid()
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Two equal echoes separated by 20 us, light noise
record_len = 1000
echo_delays = [400, 420]  # samples (= us)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

for ax, (name, s) in zip(axes, waveforms.items()):
    x = awgn(record_len, noise_power=1e-3)
    for d in echo_delays:
        x[d:d + len(s)] += s

    r = signal.correlate(x, s, mode="valid")
    lag_us = np.arange(len(r)) / fs * 1e6

    ax.plot(lag_us, np.abs(r))
    for d in echo_delays:
        ax.axvline(d, color="k", linestyle=":", linewidth=1)

    ax.set_xlim(250, 570)
    ax.set_xlabel("Delay [us]")
    ax.set_title(name)
    ax.grid()

axes[0].set_ylabel("|matched filter output|")
plt.suptitle("Two echoes 20 us apart")
plt.tight_layout()
plt.show()

### Observation

- 

## Experiment 4: Setting the Threshold

Use noise-only trials ($\mathcal{H}_0$) with $N = 128$ to:

1. Compare the histograms of the matched filter and energy detector statistics with their theoretical distributions.
2. Set both thresholds for $P_{FA} = 10^{-3}$ and measure the actual false-alarm rate.
3. Keep the thresholds fixed, but let the **real** noise power be 0.5, 1 and 2 dB higher than the value assumed when the thresholds were set.

In [ ]:
N = 128
pfa_target = 1e-3
sigma2_assumed = 1.0
n_trials = 200_000

template = bpsk_code(N)

# Thresholds from theory, using the assumed noise power
gamma_mf = -sigma2_assumed * np.log(pfa_target)
gamma_ed = sigma2_assumed / 2 * stats.chi2.isf(pfa_target, 2 * N)

# Noise-only trials
T_mf0, T_ed0 = run_trials(template, n_trials, noise_power=sigma2_assumed)

print(f"MF threshold: {gamma_mf:.2f}  -> measured Pfa = {np.mean(T_mf0 > gamma_mf):.2e}")
print(f"ED threshold: {gamma_ed:.2f} -> measured Pfa = {np.mean(T_ed0 > gamma_ed):.2e}")
print(f"Target Pfa:   {pfa_target:.0e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matched filter statistic: exponential with mean sigma^2
bins = np.linspace(0, 12, 100)
axes[0].hist(T_mf0, bins=bins, density=True, alpha=0.6, label="simulated")
axes[0].plot(bins, stats.expon.pdf(bins, scale=sigma2_assumed), "k", label="theory (exponential)")
axes[0].axvline(gamma_mf, color="tab:red", linestyle="--", label=f"threshold (Pfa = {pfa_target:.0e})")
axes[0].set_yscale("log")
axes[0].set_xlabel("T_MF")
axes[0].set_ylabel("PDF")
axes[0].set_title("Matched filter statistic under H0")
axes[0].grid()
axes[0].legend()

# Energy detector statistic: (sigma^2 / 2) * chi-square(2N)
bins = np.linspace(80, 180, 100)
pdf_ed = stats.chi2.pdf(2 * bins / sigma2_assumed, 2 * N) * 2 / sigma2_assumed
axes[1].hist(T_ed0, bins=bins, density=True, alpha=0.6, label="simulated")
axes[1].plot(bins, pdf_ed, "k", label="theory (chi-square, 2N dof)")
axes[1].axvline(gamma_ed, color="tab:red", linestyle="--", label=f"threshold (Pfa = {pfa_target:.0e})")
axes[1].set_yscale("log")
axes[1].set_xlabel("T_ED")
axes[1].set_title("Energy detector statistic under H0")
axes[1].grid()
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Real noise power higher than assumed, thresholds unchanged
noise_errors_db = [0, 0.5, 1.0, 2.0]

print(f"{'noise error [dB]':>17} {'MF Pfa':>10} {'ED Pfa':>10}")
print("-" * 40)

for err_db in noise_errors_db:
    sigma2_true = sigma2_assumed * 10 ** (err_db / 10)
    T_mf, T_ed = run_trials(template, n_trials, noise_power=sigma2_true)
    print(f"{err_db:>17.1f} {np.mean(T_mf > gamma_mf):>10.2e} {np.mean(T_ed > gamma_ed):>10.2e}")

### Observation

- 

## Experiment 5: Detection Probability vs SNR

With $N = 128$ and $P_{FA} = 10^{-3}$, sweep the input SNR per sample and measure $P_D$ for both detectors by Monte Carlo. Compare with the theoretical curves.

The receiver window is assumed to be perfectly aligned with the signal, so this is the best case for both detectors.

In [ ]:
N = 128
pfa_target = 1e-3
sigma2 = 1.0
n_trials = 4000

template = bpsk_code(N)

gamma_mf = -sigma2 * np.log(pfa_target)
gamma_ed = sigma2 / 2 * stats.chi2.isf(pfa_target, 2 * N)

snr_grid_db = np.arange(-26, 1, 2)
snr_fine_db = np.linspace(-26, 0, 200)


def pd_theory(snr_db, N, gamma_mf, gamma_ed, sigma2=1.0):
    """Theoretical Pd for the matched filter and energy detector."""
    E = N * sigma2 * 10 ** (np.asarray(snr_db) / 10)
    nc = 2 * E / sigma2
    pd_mf = stats.ncx2.sf(2 * gamma_mf / sigma2, 2, nc)
    pd_ed = stats.ncx2.sf(2 * gamma_ed / sigma2, 2 * N, nc)
    return pd_mf, pd_ed


pd_mf_sim, pd_ed_sim = [], []

for snr_db in snr_grid_db:
    amp = np.sqrt(sigma2 * 10 ** (snr_db / 10))
    T_mf, T_ed = run_trials(template, n_trials, rx_signal=amp * template, noise_power=sigma2)
    pd_mf_sim.append(np.mean(T_mf > gamma_mf))
    pd_ed_sim.append(np.mean(T_ed > gamma_ed))

pd_mf_th, pd_ed_th = pd_theory(snr_fine_db, N, gamma_mf, gamma_ed, sigma2)

plt.figure(figsize=(10, 5.5))
plt.plot(snr_fine_db, pd_mf_th, "tab:blue", label="matched filter (theory)")
plt.plot(snr_grid_db, pd_mf_sim, "o", color="tab:blue", label="matched filter (simulated)")
plt.plot(snr_fine_db, pd_ed_th, "tab:orange", label="energy detector (theory)")
plt.plot(snr_grid_db, pd_ed_sim, "s", color="tab:orange", label="energy detector (simulated)")
plt.axhline(0.9, color="k", linestyle=":", linewidth=1)
plt.xlabel("Input SNR per sample [dB]")
plt.ylabel("Probability of detection")
plt.title(f"Pd vs SNR (N = {N}, Pfa = {pfa_target:.0e})")
plt.grid()
plt.legend()
plt.show()

snr_90_mf = snr_fine_db[np.argmax(pd_mf_th >= 0.9)]
snr_90_ed = snr_fine_db[np.argmax(pd_ed_th >= 0.9)]
print(f"SNR for Pd = 0.9, matched filter:  {snr_90_mf:.1f} dB")
print(f"SNR for Pd = 0.9, energy detector: {snr_90_ed:.1f} dB")
print(f"Difference:                        {snr_90_ed - snr_90_mf:.1f} dB")

### Observation

- 

## Experiment 6: Frequency Offset

A real receiver rarely sees the signal at exactly the expected frequency: oscillator errors and Doppler add a frequency offset $f_o$.

1. Compute the matched filter output loss versus $f_o$ for the 128-sample BPSK code and for a 128-sample LFM chirp ($B$ = 200 kHz). For each waveform, evaluate the output both at the correct delay and at the lag where the output is largest.
2. At a fixed input SNR of -6 dB, measure $P_D$ versus $f_o$ for the BPSK matched filter and for the energy detector.

The offset axis is shown in units of $1/T$, where $T = N / f_s$ is the waveform duration.

In [ ]:
N = 128
T = N / fs
offsets = np.linspace(0, 4 / T, 161)  # Hz
n = np.arange(N)

s_bpsk = bpsk_code(N) / np.sqrt(N)
s_lfm = lfm_chirp(N, 200e3) / np.sqrt(N)


def mf_loss(s, f_o):
    """
    |matched filter output|^2 (unit-energy template) for a frequency-shifted copy of s.
    Returns (value at the correct delay, largest value over all lags, lag of the largest value).
    """
    rx = s * np.exp(1j * 2 * np.pi * f_o * n / fs)
    r = np.abs(signal.correlate(rx, s, mode="full")) ** 2
    lags = signal.correlation_lags(N, N, mode="full")
    i_best = np.argmax(r)
    return r[lags == 0][0], r[i_best], lags[i_best]


res_bpsk = np.array([mf_loss(s_bpsk, fo) for fo in offsets])
res_lfm = np.array([mf_loss(s_lfm, fo) for fo in offsets])

loss_bpsk = res_bpsk[:, 0]  # used later: the detector looks at the correct delay

plt.figure(figsize=(10, 5))
plt.plot(offsets * T, db(res_bpsk[:, 0]), "tab:blue", linewidth=4, alpha=0.5, label="BPSK code, correct delay")
plt.plot(offsets * T, db(res_bpsk[:, 1]), "tab:blue", linestyle="--", label="BPSK code, best lag")
plt.plot(offsets * T, db(res_lfm[:, 0]), "tab:orange", label="LFM chirp, correct delay")
plt.plot(offsets * T, db(res_lfm[:, 1]), "tab:orange", linestyle="--", label="LFM chirp, best lag")
plt.xlabel("Frequency offset x T")
plt.ylabel("Matched filter peak [dB]")
plt.title("Matched filter loss vs frequency offset")
plt.ylim(-40, 2)
plt.grid()
plt.legend(loc="upper right")
plt.show()

print(f"1/T = {1 / T / 1e3:.2f} kHz")
for k in [1, 2, 4]:
    i = np.argmin(np.abs(offsets - k / T))
    print(f"f_o = {k}/T: LFM best lag = {res_lfm[i, 2]:+.0f} samples ({res_lfm[i, 2] / fs * 1e6:+.0f} us)")

In [ ]:
snr_db = -6
sigma2 = 1.0
pfa_target = 1e-3
n_trials = 4000

amp = np.sqrt(sigma2 * 10 ** (snr_db / 10))
template = s_bpsk * np.sqrt(N)  # unit-amplitude BPSK code
E = N * amp**2

gamma_mf = -sigma2 * np.log(pfa_target)
gamma_ed = sigma2 / 2 * stats.chi2.isf(pfa_target, 2 * N)

# Theory: matched filter sees E * loss, energy detector always sees E
pd_mf_th = stats.ncx2.sf(2 * gamma_mf / sigma2, 2, 2 * E * loss_bpsk / sigma2)
pd_ed_th = np.full_like(offsets, stats.ncx2.sf(2 * gamma_ed / sigma2, 2 * N, 2 * E / sigma2))

# Monte Carlo at a few offsets
mc_offsets = np.arange(0, 4.01, 0.25) / T
pd_mf_sim, pd_ed_sim = [], []

for fo in mc_offsets:
    rx = amp * template * np.exp(1j * 2 * np.pi * fo * n / fs)
    T_mf, T_ed = run_trials(template, n_trials, rx_signal=rx, noise_power=sigma2)
    pd_mf_sim.append(np.mean(T_mf > gamma_mf))
    pd_ed_sim.append(np.mean(T_ed > gamma_ed))

plt.figure(figsize=(10, 5.5))
plt.plot(offsets * T, pd_mf_th, "tab:blue", label="matched filter (theory)")
plt.plot(mc_offsets * T, pd_mf_sim, "o", color="tab:blue", label="matched filter (simulated)")
plt.plot(offsets * T, pd_ed_th, "tab:orange", label="energy detector (theory)")
plt.plot(mc_offsets * T, pd_ed_sim, "s", color="tab:orange", label="energy detector (simulated)")
plt.xlabel("Frequency offset x T")
plt.ylabel("Probability of detection")
plt.title(f"Pd vs frequency offset (SNR = {snr_db} dB, N = {N}, Pfa = {pfa_target:.0e})")
plt.ylim(-0.02, 1.02)
plt.grid()
plt.legend()
plt.show()

### Observation

- 

## Key Takeaways

-

## Next Step

The next notebook will investigate adaptive thresholds (CFAR), focusing on how the detection threshold can follow an unknown and changing noise floor while keeping the false-alarm rate constant.